# 07 — Temporal and Textual Alignment of Facebook Discourse and State Legislation


    **Research objectives.** Describe whether state-specific Facebook attention occurs before, during, or after recorded legislative events, assess textual alignment between bills and posts, and evaluate whether strictly prior state-level attention adds out-of-sample predictive information to the legislative baseline.

    **Inputs.** Scored bills and the prepared consistent `datacenter` corpus; the April–August 2026 subset is used for the frame-prevalence comparison.

    **Methods.** Apply full-state-name regular expressions without ambiguous abbreviations; reshape to one row per post-state pair; construct complete state-month panels with explicit zeros; estimate introduction and latest-passage event windows from −3 to +3 months; report all Spearman lags from −3 to +3; apply shared frame dictionaries; select case-study states using prespecified activity diagnostics; and audit strictly pre-introduction social features. The augmented model runs only when at least 150 bills have 90 days of prior coverage, at least 10% have nonzero prior attention, and each outcome category contains at least 30 cases.

    **Outputs.** State-month heatmaps, national timeline, event-window figure/tables, lag table, frame alignment, case-state selection, prior-attention bill file, and baseline-vs-social metrics only when the prespecified feasibility rules pass.

In [1]:
from __future__ import annotations

import json

import math

import re

from pathlib import Path

import matplotlib

import matplotlib.dates as mdates

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer

from sklearn.decomposition import LatentDirichletAllocation

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

matplotlib.use("Agg")

SEED = 149

COLORS = {
    "restrictive": "#D55E00",
    "neutral": "#7A7A7A",
    "supportive": "#009E73",
    "blue": "#0072B2",
    "orange": "#E69F00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
}

STATES = [
    "California", "Georgia", "Illinois", "Indiana", "Kentucky", "Maine",
    "Maryland", "Minnesota", "New York", "North Dakota", "Ohio", "Oklahoma",
    "Oregon", "Texas", "Virginia",
]

REGIONS = {
    "California": "West", "Oregon": "West",
    "Illinois": "Midwest", "Indiana": "Midwest", "Minnesota": "Midwest",
    "North Dakota": "Midwest", "Ohio": "Midwest",
    "Georgia": "South", "Kentucky": "South", "Maryland": "South",
    "Oklahoma": "South", "Texas": "South", "Virginia": "South",
    "Maine": "Northeast", "New York": "Northeast",
}

FRAME_PATTERNS = {
    "Energy and utility costs": [
        r"\belectric(?:ity|al)\b", r"\bpower\b", r"\bgrid\b", r"\butilit(?:y|ies)\b",
        r"\bratepayers?\b", r"\brates?\b", r"\bmegawatts?\b", r"\btransmission\b",
        r"\binterconnection\b", r"\benergy\b",
    ],
    "Water and environmental effects": [
        r"\bwater\b", r"\benvironment(?:al)?\b", r"\bemissions?\b", r"\bcarbon\b",
        r"\bpollution\b", r"\bclimate\b", r"\bsustainab(?:le|ility)\b", r"\baquifer\b",
    ],
    "Economic development and jobs": [
        r"\bjobs?\b", r"\bemployment\b", r"\beconomic development\b", r"\binvest(?:ment|s|ed)\b",
        r"\bbusiness(?:es)?\b", r"\bconstruction\b", r"\bgrowth\b", r"\brevenue\b",
    ],
    "Taxes, incentives, and subsidies": [
        r"\btax(?:es|ation)?\b", r"\btax (?:credit|break|exemption)s?\b", r"\bincentives?\b",
        r"\bsubsid(?:y|ies)\b", r"\babatement\b", r"\bpublic funds?\b",
    ],
    "Regulation and community control": [
        r"\bregulat(?:ion|e|ed|ory)\b", r"\bzoning\b", r"\bpermits?\b", r"\bordina(?:nce|nces)\b",
        r"\bmoratorium\b", r"\bbans?\b", r"\bcommunity\b", r"\blocal control\b",
        r"\bpublic hearings?\b", r"\bdisclos(?:ure|e)\b", r"\btransparency\b",
    ],
    "AI growth and technological competition": [
        r"\bartificial intelligence\b", r"\bAI\b", r"\bcloud\b", r"\bcompute\b",
        r"\bdigital infrastructure\b", r"\btechnology\b", r"\binnovation\b", r"\bhyperscale\b",
    ],
}

FRAME_REGEX = {
    name: re.compile("|".join(patterns), re.IGNORECASE)
    for name, patterns in FRAME_PATTERNS.items()
}

def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")

def save_gzip(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, compression="gzip")
    print(f"Saved {len(frame):,} rows -> {path}")

def save_figure(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved figure -> {path}")

def add_frame_flags(frame: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    out = frame.copy()
    text = out[text_col].fillna("").astype(str)
    for name, regex in FRAME_REGEX.items():
        out[f"frame_{slug(name)}"] = text.str.contains(regex, na=False)
    return out

def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

def load_scored_bills(path: Path) -> pd.DataFrame:
    bills = pd.read_csv(path, low_memory=False)
    for column in ["first_action_date", "latest_action_date", "latest_passage_date"]:
        bills[column] = pd.to_datetime(bills[column], errors="coerce", utc=True)
    numeric = [
        "supportiveness_score", "supportiveness_confidence", "sponsor_count",
        "primary_sponsor_count", "passage_count", "passed_vote_event_count",
    ]
    for column in numeric:
        bills[column] = pd.to_numeric(bills[column], errors="coerce")
    bills["advanced"] = (
        bills["derived_status"].isin(["passed_chamber_or_legislature", "enacted", "vetoed"])
        | bills["passage_count"].fillna(0).gt(0)
        | bills["passed_vote_event_count"].fillna(0).gt(0)
    ).astype(int)
    bills["score_numeric"] = bills["supportiveness_score"]
    bills["introduction_year"] = bills["first_action_date"].dt.year.astype("Int64")
    bills["region"] = bills["state"].map(REGIONS).fillna("Other")
    return bills

def scored_analysis_sample(bills: pd.DataFrame) -> pd.DataFrame:
    mask = (
        bills["dc_relevant"].fillna("").str.lower().eq("yes")
        & bills["supportiveness_status"].fillna("").str.lower().eq("scored")
        & bills["score_numeric"].between(1, 10)
    )
    out = bills.loc[mask].copy()
    out["supportiveness_group"] = pd.cut(
        out["score_numeric"], bins=[0, 3, 6, 10],
        labels=["Restrictive (1-3)", "Neutral or mixed (4-6)", "Supportive (7-10)"],
    )
    return out

def make_bill_model_pipeline(numeric: list[str], categorical: list[str], class_weight="balanced") -> Pipeline:
    preprocess = ColumnTransformer([
        ("numeric", Pipeline([("scale", StandardScaler())]), numeric),
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=SEED)),
    ])

def classification_metrics(y_true, y_pred, y_prob, model_name: str) -> dict:
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "n": len(y_true), "positive_n": int(np.sum(y_true)),
    }

def evaluate_bill_models(data: pd.DataFrame, social_columns: list[str] | None = None) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    numeric = ["score_numeric", "sponsor_count", "primary_sponsor_count", "introduction_year"]
    if social_columns:
        numeric += social_columns
    categorical = ["originating_chamber", "region"]
    keep = ["advanced"] + numeric + categorical
    model_data = data[keep].copy()
    for column in numeric:
        model_data[column] = pd.to_numeric(model_data[column], errors="coerce")
        model_data[column] = model_data[column].fillna(model_data[column].median()).fillna(0)
    model_data[categorical] = model_data[categorical].fillna("Missing")
    X, y = model_data[numeric + categorical], model_data.advanced.astype(int)
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    pipeline = make_bill_model_pipeline(numeric, categorical)
    pred = cross_val_predict(pipeline, X, y, cv=folds, method="predict")
    prob = cross_val_predict(pipeline, X, y, cv=folds, method="predict_proba")[:, 1]
    majority = int(y.mean() >= .5)
    base_pred = np.repeat(majority, len(y))
    base_prob = np.repeat(y.mean(), len(y))
    label = "Social-augmented legislative model" if social_columns else "Baseline legislative model"
    metrics = pd.DataFrame([
        classification_metrics(y, base_pred, base_prob, "Majority-class baseline"),
        classification_metrics(y, pred, prob, label),
    ])
    return metrics, y.to_numpy(), pred, prob

def _read_processed(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    if "creation_time" in frame:
        frame["creation_time"] = pd.to_datetime(frame["creation_time"], errors="coerce", utc=True)
    bool_columns = [c for c in frame if c == "policy_related" or c.startswith("frame_")]
    for column in bool_columns:
        if frame[column].dtype != bool:
            frame[column] = frame[column].astype(str).str.lower().eq("true")
    return frame

def state_pairs(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for record in frame[["id","creation_time","text","policy_related","post_owner.id","total_engagement"]].itertuples(index=False, name=None):
        post_id, date, text, policy, owner, engagement = record
        value = "" if pd.isna(text) else str(text)
        matches = [state for state in STATES if re.search(rf"\b{re.escape(state)}\b", value, re.IGNORECASE)]
        for state in matches:
            rows.append({"id": post_id, "creation_time": date, "state": state, "policy_related": policy, "post_owner.id": owner, "total_engagement": engagement, "state_mentions_in_post": len(matches), "text": value, "potential_false_match_reason": "Georgia may refer to the country" if state == "Georgia" else "state mention is not evidence of post origin"})
    pairs = pd.DataFrame(rows)
    matched_ids = set(pairs.id) if len(pairs) else set()
    diagnostics = pd.DataFrame([{
        "relevant_posts": len(frame), "posts_mentioning_study_state": len(matched_ids),
        "percent_mentioning_study_state": 100*len(matched_ids)/len(frame) if len(frame) else np.nan,
        "multi_state_posts": pairs.loc[pairs.state_mentions_in_post.gt(1), "id"].nunique() if len(pairs) else 0,
        "unmatched_posts": len(frame)-len(matched_ids), "post_state_pairs": len(pairs),
    }])
    return pairs, diagnostics

def build_social_features(bills: pd.DataFrame, policy_pairs: pd.DataFrame, longitudinal: pd.DataFrame) -> pd.DataFrame:
    text_frames = longitudinal.set_index("id")[[c for c in longitudinal if c.startswith("frame_")]]
    print(
        f"Before post-frame merge: policy state-pairs={len(policy_pairs):,}; "
        f"unique post IDs={policy_pairs['id'].nunique():,}; frame rows={len(text_frames):,}"
    )
    pairs = policy_pairs.merge(
        text_frames, left_on="id", right_index=True, how="left",
        validate="many_to_one", indicator="_frame_merge",
    )
    matched = int(pairs["_frame_merge"].eq("both").sum())
    print(f"After post-frame merge: {len(pairs):,} rows; matched={matched:,}; unmatched={len(pairs)-matched:,}")
    pairs = pairs.drop(columns="_frame_merge")
    rows = []
    start = pd.Timestamp("2024-01-01", tz="UTC")
    end = pd.Timestamp("2026-08-17 23:59:59", tz="UTC")
    eligible_bills = bills.loc[bills.first_action_date.between(start + pd.Timedelta(days=90), end)].copy()
    for bill in eligible_bills.itertuples():
        state_posts = pairs.loc[pairs.state.eq(bill.state)]
        prior90 = state_posts.loc[state_posts.creation_time.lt(bill.first_action_date) & state_posts.creation_time.ge(bill.first_action_date-pd.Timedelta(days=90))]
        prior30 = prior90.loc[prior90.creation_time.ge(bill.first_action_date-pd.Timedelta(days=30))]
        combined_frame_cols = [f"frame_{slug(x)}" for x in ["Energy and utility costs","Water and environmental effects","Regulation and community control"]]
        any_concern = prior90[combined_frame_cols].any(axis=1).mean() if len(prior90) else 0
        rows.append({
            "bill_id": bill.bill_id, "prior_30_policy_posts": prior30.id.nunique(),
            "prior_90_policy_posts": prior90.id.nunique(), "prior_90_unique_owners": prior90["post_owner.id"].nunique(),
            "prior_90_median_engagement": prior90.total_engagement.median() if len(prior90) else 0,
            "prior_90_concern_frame_share": any_concern,
        })
    feature_rows = pd.DataFrame(rows)
    print(
        f"Before bill-feature merge: eligible bills={len(eligible_bills):,}; "
        f"feature rows={len(feature_rows):,}"
    )
    merged = eligible_bills.merge(
        feature_rows, on="bill_id", how="left", validate="one_to_one",
        indicator="_feature_merge",
    )
    matched = int(merged["_feature_merge"].eq("both").sum())
    print(f"After bill-feature merge: {len(merged):,} rows; matched={matched:,}; unmatched={len(merged)-matched:,}")
    return merged.drop(columns="_feature_merge")

def run_linkage(root: Path) -> dict:
    bills = scored_analysis_sample(load_scored_bills(root / "data/processed/bills_scored.csv"))
    longitudinal = _read_processed(root / "data/processed/facebook_datacenter_longitudinal_2024_2026.csv.gz")
    pairs, state_diag = state_pairs(longitudinal)
    policy_pairs = pairs.loc[pairs.policy_related].copy()
    print(f"Longitudinal posts: {len(longitudinal):,}; state-matched posts: {pairs.id.nunique():,}; policy state-pairs: {len(policy_pairs):,}")
    save_table(state_diag, root / "output/tables/facebook_state_mention_diagnostics.csv")
    if len(pairs):
        pairs["year"] = pairs.creation_time.dt.year
        state_counts = pairs.groupby(["state","year"]).agg(posts=("id","nunique"), policy_posts=("policy_related","sum")).reset_index()
    else:
        state_counts = pd.DataFrame(columns=["state","year","posts","policy_posts"])
    save_table(state_counts, root / "output/tables/facebook_state_mentions_by_year.csv")
    if len(pairs):
        state_review = pairs.sample(n=min(75, len(pairs)), random_state=SEED)[[
            "id","creation_time","state","state_mentions_in_post","potential_false_match_reason","text"
        ]].copy()
        state_review["manual_state_context_valid"] = ""
        state_review["manual_notes"] = ""
        save_table(state_review, root / "output/tables/facebook_state_mention_manual_review.csv")

    events = pd.concat([
        bills[["bill_id","state","first_action_date"]].rename(columns={"first_action_date":"event_date"}).assign(event_type="introduction"),
        bills[["bill_id","state","latest_passage_date"]].rename(columns={"latest_passage_date":"event_date"}).assign(event_type="latest_recorded_passage"),
        bills[["bill_id","state","latest_action_date"]].rename(columns={"latest_action_date":"event_date"}).assign(event_type="latest_action_proxy"),
    ], ignore_index=True).dropna(subset=["event_date"]).drop_duplicates(["bill_id","event_type","event_date"])
    events = events.loc[events.event_date.between("2024-01-01", "2026-08-17 23:59:59+00:00")].copy()
    save_table(events, root / "output/tables/legislative_events.csv")

    months = pd.period_range("2024-01", "2026-08", freq="M").astype(str)
    panel = pd.MultiIndex.from_product([STATES, months], names=["state","month"]).to_frame(index=False)
    if len(policy_pairs):
        policy_pairs["month"] = policy_pairs.creation_time.dt.tz_localize(None).dt.to_period("M").astype(str)
        fb_month = policy_pairs.groupby(["state","month"]).agg(facebook_policy_posts=("id","nunique"), facebook_unique_owners=("post_owner.id","nunique")).reset_index()
    else:
        fb_month = pd.DataFrame(columns=["state","month","facebook_policy_posts","facebook_unique_owners"])
    events["month"] = events.event_date.dt.tz_localize(None).dt.to_period("M").astype(str)
    event_month = events.groupby(["state","month","event_type"]).bill_id.nunique().unstack(fill_value=0).reset_index()
    print(f"Panel skeleton rows before merges: {len(panel):,}")
    panel = panel.merge(fb_month, on=["state","month"], how="left", validate="one_to_one")
    print(f"After Facebook merge: {len(panel):,}; matched nonzero rows: {panel.facebook_policy_posts.notna().sum():,}")
    panel = panel.merge(event_month, on=["state","month"], how="left", validate="one_to_one")
    print(f"After event merge: {len(panel):,}")
    numeric = [c for c in panel if c not in ["state","month"]]
    panel[numeric] = panel[numeric].fillna(0)
    save_table(panel, root / "output/tables/state_month_panel.csv")

    order = bills.groupby("state").size().sort_values().index.tolist()
    matrices = []
    for field in ["facebook_policy_posts","introduction","latest_recorded_passage"]:
        matrices.append(panel.pivot(index="state", columns="month", values=field).reindex(order).fillna(0))
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    cmaps = ["Blues","Greens","Oranges"]
    titles = ["State-matched Facebook policy posts", "Bill introductions", "Latest recorded passage events"]
    for ax, matrix, cmap, title in zip(axes, matrices, cmaps, titles):
        im = ax.imshow(matrix, aspect="auto", cmap=cmap)
        ax.set_yticks(range(len(order)), order, fontsize=8)
        ax.set_title(title, loc="left", fontsize=11)
        fig.colorbar(im, ax=ax, pad=.01, shrink=.8)
    axes[-1].set_xticks(range(0,len(months),2), [months[i] for i in range(0,len(months),2)], rotation=45, ha="right")
    fig.suptitle("State-Month Alignment of Facebook Policy Discourse and Legislative Activity, January 2024–August 2026\nFacebook measures use posts returned by the 'datacenter' query; state mentions do not establish post origin")
    fig.tight_layout()
    save_figure(fig, root / "output/figures/state_month_discourse_legislation_heatmap.png")

    longitudinal["month"] = longitudinal.creation_time.dt.tz_localize(None).dt.to_period("M").astype(str)
    national_fb = longitudinal.groupby("month").agg(relevant_posts=("id","nunique"), policy_posts=("policy_related","sum"), unique_owners=("post_owner.id","nunique")).reset_index()
    national_events = events.loc[events.event_type.isin(["introduction","latest_recorded_passage"])].groupby(["month","event_type"]).bill_id.nunique().unstack(fill_value=0).reset_index()
    national = pd.DataFrame({"month": months})
    print(
        f"Before national timeline merges: calendar={len(national):,}; "
        f"Facebook months={len(national_fb):,}; event months={len(national_events):,}"
    )
    national = national.merge(
        national_fb, on="month", how="left", validate="one_to_one",
        indicator="_facebook_merge",
    )
    facebook_matches = int(national["_facebook_merge"].eq("both").sum())
    national = national.drop(columns="_facebook_merge")
    national = national.merge(
        national_events, on="month", how="left", validate="one_to_one",
        indicator="_event_merge",
    )
    event_matches = int(national["_event_merge"].eq("both").sum())
    national = national.drop(columns="_event_merge").fillna(0)
    print(
        f"After national timeline merges: {len(national):,} rows; "
        f"Facebook matches={facebook_matches:,}; event matches={event_matches:,}"
    )
    save_table(national, root / "output/tables/national_monthly_timeline.csv")
    fig, ax1 = plt.subplots(figsize=(13,6))
    dates = pd.to_datetime(national.month)
    ax1.plot(dates,national.relevant_posts,label="Relevant posts",color=COLORS["blue"])
    ax1.plot(dates,national.policy_posts,label="Policy posts",color=COLORS["orange"])
    ax1.plot(dates,national.unique_owners,label="Unique owners",color=COLORS["purple"])
    ax2=ax1.twinx()
    ax2.bar(dates-pd.Timedelta(days=5),national.get("introduction",0),width=8,alpha=.28,color=COLORS["supportive"],label="Introductions")
    ax2.bar(dates+pd.Timedelta(days=5),national.get("latest_recorded_passage",0),width=8,alpha=.35,color=COLORS["restrictive"],label="Recorded passages")
    ax1.set_ylabel("Facebook count (consistent 'datacenter' sample)"); ax2.set_ylabel("Bill events")
    ax1.set_title("National Monthly Facebook Attention and Recorded Legislative Events, January 2024–August 2026")
    lines,labels=ax1.get_legend_handles_labels(); lines2,labels2=ax2.get_legend_handles_labels(); ax1.legend(lines+lines2,labels+labels2,ncol=3,fontsize=8)
    fig.tight_layout(); save_figure(fig, root / "output/figures/national_monthly_timeline.png")

    event_windows=[]
    focal = events.loc[events.event_type.isin(["introduction","latest_recorded_passage"])].copy()
    for event in focal.itertuples():
        event_period = event.event_date.tz_localize(None).to_period("M")
        state_series = panel.loc[panel.state.eq(event.state)].set_index("month").facebook_policy_posts
        values={}
        for lag in range(-3,4):
            month=str(event_period+lag); value=float(state_series.get(month,0)); values[lag]=value
            event_windows.append({"bill_id":event.bill_id,"state":event.state,"event_type":event.event_type,"event_date":event.event_date,"relative_month":lag,"facebook_policy_posts":value})
    window_detail=pd.DataFrame(event_windows)
    summary=window_detail.groupby(["event_type","relative_month"]).agg(events=("bill_id","nunique"),events_with_any_posts=("facebook_policy_posts",lambda x:int((x>0).sum())),median_posts=("facebook_policy_posts","median"),mean_posts=("facebook_policy_posts","mean"),standard_error=("facebook_policy_posts",lambda x:x.std(ddof=1)/np.sqrt(len(x)) if len(x)>1 else 0)).reset_index()
    save_table(window_detail, root / "output/tables/legislative_event_window_detail.csv")
    save_table(summary, root / "output/tables/legislative_event_window_summary.csv")
    fig,ax=plt.subplots(figsize=(9,6))
    for event_type,color in [("introduction",COLORS["blue"]),("latest_recorded_passage",COLORS["orange"])]:
        x=summary.loc[summary.event_type.eq(event_type)]
        ax.errorbar(x.relative_month,x.mean_posts,yerr=1.96*x.standard_error,marker="o",capsize=3,label=f"{event_type.replace('_',' ').title()} (events={x.events.max():.0f})",color=color)
    ax.axvline(0,color="black",ls="--",lw=1); ax.set_xticks(range(-3,4)); ax.set_xlabel("Month relative to recorded legislative event"); ax.set_ylabel("Mean state-matched Facebook policy-post count"); ax.set_title("Facebook Policy Discourse in Legislative Event Windows\nPosts returned by the 'datacenter' query; error bars are approximate 95% intervals"); ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
    save_figure(fig,root/"output/figures/legislative_event_window.png")

    lag_rows=[]
    for lag in range(-3,4):
        shifted=panel.groupby("state").facebook_policy_posts.shift(lag)
        for event_field in ["introduction","latest_recorded_passage"]:
            rho,p=spearmanr(shifted,panel.get(event_field,pd.Series(0,index=panel.index)),nan_policy="omit")
            lag_rows.append({"lag":lag,"definition":"positive lag means Facebook count from earlier months is paired with current events","event_type":event_field,"spearman_rho":rho,"p_value_descriptive":p,"panel_rows":len(panel)})
    save_table(pd.DataFrame(lag_rows),root/"output/tables/lead_lag_spearman.csv")

    bill_text=bills.assign(text=bills.title.fillna("")+" "+bills.abstract.fillna(""))
    bill_text=add_frame_flags(bill_text)
    datasets={"All Scored Bills":bill_text,"Bills Introduced, 2024–2026":bill_text.loc[bill_text.first_action_date.ge("2024-01-01")],"Facebook Posts, 'datacenter' Query, 2024–2026":add_frame_flags(longitudinal),"Facebook Posts, 'datacenter' Query, April–August 2026":add_frame_flags(_read_processed(root/"data/processed/facebook_datacenter_discourse_2026.csv.gz"))}
    alignment=[]
    for sample,data in datasets.items():
        for frame_name in FRAME_PATTERNS:
            col=f"frame_{slug(frame_name)}"; alignment.append({"sample":sample,"frame":frame_name,"n":len(data),"percent":100*data[col].mean()})
    alignment=pd.DataFrame(alignment); save_table(alignment,root/"output/tables/frame_alignment_bills_facebook.csv")
    pivot=alignment.pivot(index="frame",columns="sample",values="percent")
    fig,ax=plt.subplots(figsize=(12,7)); pivot.plot(kind="barh",ax=ax,color=[COLORS["blue"],COLORS["sky"],COLORS["orange"],COLORS["purple"]]); ax.set_xlabel("Percentage containing frame language (categories may overlap)"); ax.set_title("Policy-Frame Prevalence in Legislative Text and Facebook Posts\nShared regex dictionaries; textual similarity does not establish diffusion or influence"); ax.grid(axis="x",alpha=.2); fig.tight_layout(); save_figure(fig,root/"output/figures/frame_alignment_bills_facebook.png")

    # Estimate exploratory state- and time-matched textual alignment in a shared
    # TF-IDF feature space. Each bill is compared only with state-matched policy
    # posts published within 90 days before or after its first recorded action.
    post_lookup = longitudinal.drop_duplicates("id").set_index("id")
    candidate_ids = policy_pairs.id.drop_duplicates().tolist()
    bill_docs = bill_text.text.fillna("").tolist()
    post_docs = post_lookup.loc[candidate_ids, "normalized_text"].fillna("").tolist()
    similarity_rows = []
    if bill_docs and post_docs:
        similarity_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_features=8000)
        similarity_matrix = similarity_vectorizer.fit_transform(bill_docs + post_docs)
        bill_vectors = similarity_matrix[:len(bill_docs)]
        post_vectors = similarity_matrix[len(bill_docs):]
        post_position = {post_id: i for i, post_id in enumerate(candidate_ids)}
        for bill_position, bill in enumerate(bill_text.itertuples()):
            if pd.isna(bill.first_action_date):
                continue
            candidates = policy_pairs.loc[
                policy_pairs.state.eq(bill.state)
                & policy_pairs.creation_time.between(bill.first_action_date-pd.Timedelta(days=90), bill.first_action_date+pd.Timedelta(days=90))
            ].drop_duplicates("id")
            if candidates.empty:
                continue
            positions = [post_position[x] for x in candidates.id if x in post_position]
            scores = (post_vectors[positions] @ bill_vectors[bill_position].T).toarray().ravel()
            best_local = int(np.argmax(scores))
            best_post_id = candidates.iloc[best_local].id
            best_post = post_lookup.loc[best_post_id]
            similarity_rows.append({
                "bill_id": bill.bill_id, "state": bill.state, "identifier": bill.identifier,
                "first_action_date": bill.first_action_date, "bill_title": bill.title,
                "post_id": best_post_id, "post_date": best_post.creation_time,
                "cosine_similarity": float(scores[best_local]), "post_text": str(best_post.text)[:1500],
                "manual_alignment_valid": "", "manual_notes": "",
            })
    similarity = pd.DataFrame(similarity_rows).sort_values("cosine_similarity", ascending=False) if similarity_rows else pd.DataFrame()
    save_table(similarity.head(50), root / "output/tables/bill_facebook_tfidf_alignment_top_pairs.csv")

    activity=bills.groupby("state").size().rename("bills").to_frame().join(policy_pairs.groupby("state").id.nunique().rename("facebook_policy_posts"),how="left").fillna(0)
    activity["combined_rank"]=activity.bills.rank(pct=True)+activity.facebook_policy_posts.rank(pct=True)
    cases=activity.sort_values("combined_rank",ascending=False).head(3).reset_index(); save_table(cases,root/"output/tables/state_case_study_selection.csv")
    case_states = cases.state.tolist()
    case_bills = bills.loc[bills.state.isin(case_states), [
        "bill_id","state","identifier","title","score_numeric","advanced","derived_status",
        "first_action_date","latest_passage_date","latest_action_date"
    ]].sort_values(["state","first_action_date"])
    save_table(case_bills, root / "output/tables/state_case_study_bills.csv")
    case_posts = policy_pairs.loc[policy_pairs.state.isin(case_states)].drop_duplicates(["state","id"]).copy()
    case_examples = case_posts.sort_values(["state","total_engagement"], ascending=[True,False]).groupby("state").head(5)
    save_table(case_examples[["state","id","creation_time","total_engagement","text"]], root / "output/tables/state_case_study_high_engagement_posts.csv")
    case_frame_lookup = longitudinal.drop_duplicates("id")[["id"]+[f"frame_{slug(x)}" for x in FRAME_PATTERNS]]
    print(
        f"Before case-post frame merge: case post-state rows={len(case_posts):,}; "
        f"frame lookup rows={len(case_frame_lookup):,}"
    )
    case_post_full = case_posts.merge(
        case_frame_lookup, on="id", how="left", validate="many_to_one",
        indicator="_case_frame_merge",
    )
    case_matches = int(case_post_full["_case_frame_merge"].eq("both").sum())
    print(
        f"After case-post frame merge: {len(case_post_full):,} rows; "
        f"matched={case_matches:,}; unmatched={len(case_post_full)-case_matches:,}"
    )
    case_post_full = case_post_full.drop(columns="_case_frame_merge")
    case_frame_rows=[]
    for state in case_states:
        group=case_post_full.loc[case_post_full.state.eq(state)]
        for frame_name in FRAME_PATTERNS:
            column=f"frame_{slug(frame_name)}"
            case_frame_rows.append({"state":state,"frame":frame_name,"policy_posts":group.id.nunique(),"percent_matching":100*group[column].mean() if len(group) else np.nan})
    save_table(pd.DataFrame(case_frame_rows), root / "output/tables/state_case_study_frames.csv")
    fig,axes=plt.subplots(len(case_states),1,figsize=(14,9),sharex=True)
    for ax,state in zip(np.atleast_1d(axes),case_states):
        state_panel=panel.loc[panel.state.eq(state)].copy(); x=pd.to_datetime(state_panel.month)
        ax.plot(x,state_panel.facebook_policy_posts,color=COLORS["blue"],marker="o",ms=3,label="State-matched policy posts")
        intro_dates=events.loc[events.state.eq(state)&events.event_type.eq("introduction"),"event_date"]
        passage_dates=events.loc[events.state.eq(state)&events.event_type.eq("latest_recorded_passage"),"event_date"]
        for date in intro_dates: ax.axvline(date,color=COLORS["supportive"],alpha=.14,lw=1)
        for date in passage_dates: ax.axvline(date,color=COLORS["restrictive"],alpha=.45,lw=1.2)
        ax.set_ylabel(state+"\nposts")
        ax.grid(axis="y",alpha=.2)
    axes[0].plot([],[],color=COLORS["supportive"],label="Introduction")
    axes[0].plot([],[],color=COLORS["restrictive"],label="Latest recorded passage")
    axes[0].legend(ncol=3,fontsize=8,frameon=False)
    axes[-1].set_xlabel("Month")
    fig.suptitle("Selected State Case Studies: Facebook Policy Discourse and Recorded Legislative Events\nFacebook measures use posts returned by the 'datacenter' query")
    fig.tight_layout(); save_figure(fig,root/"output/figures/state_case_studies.png")

    social=build_social_features(bills,policy_pairs,longitudinal)
    nonzero_share=social.prior_90_policy_posts.gt(0).mean() if len(social) else 0
    criteria=pd.DataFrame([{"eligible_bills":len(social),"nonzero_prior_attention_share":nonzero_share,"advanced_bills":int(social.advanced.sum()),"nonadvanced_bills":int((1-social.advanced).sum()),"eligible_at_least_150":len(social)>=150,"nonzero_share_at_least_10_percent":nonzero_share>=.10,"both_outcomes_at_least_30":social.advanced.sum()>=30 and (1-social.advanced).sum()>=30}])
    save_table(criteria,root/"output/tables/social_model_feasibility.csv")
    social_path=root/"data/processed/bills_with_prior_facebook_features.csv.gz"; save_gzip(social,social_path)
    feasible=bool(criteria[["eligible_at_least_150","nonzero_share_at_least_10_percent","both_outcomes_at_least_30"]].all(axis=None))
    social_metrics=[]
    if feasible:
        baseline_metrics,_,_,_=evaluate_bill_models(social)
        columns=["prior_30_policy_posts","prior_90_policy_posts","prior_90_unique_owners","prior_90_median_engagement","prior_90_concern_frame_share"]
        augmented_metrics,_,_,_=evaluate_bill_models(social,social_columns=columns)
        social_metrics=pd.concat([baseline_metrics.loc[baseline_metrics.model.str.startswith("Baseline")],augmented_metrics.loc[augmented_metrics.model.str.startswith("Social")]],ignore_index=True)
        save_table(social_metrics,root/"output/tables/baseline_vs_social_model_metrics.csv")
    return {"state_mentions":state_diag.to_dict("records")[0],"events":len(events),"event_windows":len(window_detail),"case_states":case_states,"text_alignment_pairs":len(similarity),"social_model_feasible":feasible,"social_metrics":social_metrics.to_dict("records") if feasible else []}

ROOT = Path("..")
print("Random seed:", SEED)

Random seed: 149


In [2]:
results = run_linkage(ROOT)
results

Longitudinal posts: 24,915; state-matched posts: 1,805; policy state-pairs: 1,094
Saved 1 rows -> ../output/tables/facebook_state_mention_diagnostics.csv
Saved 40 rows -> ../output/tables/facebook_state_mentions_by_year.csv
Saved 75 rows -> ../output/tables/facebook_state_mention_manual_review.csv
Saved 697 rows -> ../output/tables/legislative_events.csv
Panel skeleton rows before merges: 480
After Facebook merge: 480; matched nonzero rows: 136
After event merge: 480
Saved 480 rows -> ../output/tables/state_month_panel.csv


Saved figure -> ../output/figures/state_month_discourse_legislation_heatmap.png
Before national timeline merges: calendar=32; Facebook months=32; event months=24
After national timeline merges: 32 rows; Facebook matches=32; event matches=24
Saved 32 rows -> ../output/tables/national_monthly_timeline.csv


Saved figure -> ../output/figures/national_monthly_timeline.png
Saved 2,597 rows -> ../output/tables/legislative_event_window_detail.csv
Saved 14 rows -> ../output/tables/legislative_event_window_summary.csv


Saved figure -> ../output/figures/legislative_event_window.png
Saved 14 rows -> ../output/tables/lead_lag_spearman.csv


Saved 24 rows -> ../output/tables/frame_alignment_bills_facebook.csv


Saved figure -> ../output/figures/frame_alignment_bills_facebook.png


Saved 50 rows -> ../output/tables/bill_facebook_tfidf_alignment_top_pairs.csv
Saved 3 rows -> ../output/tables/state_case_study_selection.csv
Saved 184 rows -> ../output/tables/state_case_study_bills.csv
Saved 15 rows -> ../output/tables/state_case_study_high_engagement_posts.csv
Before case-post frame merge: case post-state rows=300; frame lookup rows=24,915
After case-post frame merge: 300 rows; matched=300; unmatched=0
Saved 18 rows -> ../output/tables/state_case_study_frames.csv


Saved figure -> ../output/figures/state_case_studies.png
Before post-frame merge: policy state-pairs=1,094; unique post IDs=850; frame rows=24,915
After post-frame merge: 1,094 rows; matched=1,094; unmatched=0
Before bill-feature merge: eligible bills=268; feature rows=268
After bill-feature merge: 268 rows; matched=268; unmatched=0
Saved 1 rows -> ../output/tables/social_model_feasibility.csv


Saved 268 rows -> ../data/processed/bills_with_prior_facebook_features.csv.gz
Saved 2 rows -> ../output/tables/baseline_vs_social_model_metrics.csv


{'state_mentions': {'relevant_posts': 24915,
  'posts_mentioning_study_state': 1805,
  'percent_mentioning_study_state': 7.244631747943006,
  'multi_state_posts': 156,
  'unmatched_posts': 23110,
  'post_state_pairs': 2118},
 'events': 697,
 'event_windows': 2597,
 'case_states': ['Virginia', 'Georgia', 'New York'],
 'text_alignment_pairs': 233,
 'social_model_feasible': True,
 'social_metrics': [{'model': 'Baseline legislative model',
   'accuracy': 0.6828358208955224,
   'precision': 0.45081967213114754,
   'recall': 0.7534246575342466,
   'f1': 0.5641025641025641,
   'roc_auc': 0.7870741131015104,
   'n': 268,
   'positive_n': 73},
  {'model': 'Social-augmented legislative model',
   'accuracy': 0.7276119402985075,
   'precision': 0.5,
   'recall': 0.7123287671232876,
   'f1': 0.5875706214689266,
   'roc_auc': 0.8088514225500527,
   'n': 268,
   'positive_n': 73}]}

## Interpretation, inferential scope, and limitations

A state mention does not prove post origin. `latest_action_date` is labeled as a proxy, not a full event history, and the passage date is only the latest recorded passage. Temporal alignment, lag correlations, shared language, and predictive improvement do not establish causality, influence, or diffusion. The social model is exploratory and uses only posts strictly before introduction.